# 1. Obtención de los datos

Carga de los datos originales de las dos fuentes y primera revisión: filas, columnas, tipos y fecha de obtención.

**Responsables:** Alex (Adzuna) y Vahab (Tecnoempleo).

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / 'src'))
from config import RAW_ADZUNA, RAW_TECNOEMPLEO, PROCESSED, REFERENCE

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## 1.1 Adzuna

Los JSON originales están en `data/raw/adzuna/<fecha>/search/`. Cada archivo es una consulta tecnología × ciudad × página.

In [ ]:
def cargar_adzuna(carpeta_fecha: Path) -> pd.DataFrame:
    filas = []
    for archivo in sorted((carpeta_fecha / 'search').glob('*.json')):
        tecnologia, ciudad, _ = archivo.stem.split('__')
        for oferta in json.loads(archivo.read_text(encoding='utf-8')).get('results', []):
            filas.append({
                'id_oferta': oferta.get('id'),
                'titulo': oferta.get('title'),
                'empresa': oferta.get('company', {}).get('display_name'),
                'ubicacion': oferta.get('location', {}).get('display_name'),
                'salario_min': oferta.get('salary_min'),
                'salario_max': oferta.get('salary_max'),
                'salario_estimado': str(oferta.get('salary_is_predicted')) == '1',
                'tipo_contrato': oferta.get('contract_type'),
                'jornada': oferta.get('contract_time'),
                'fecha_publicacion': oferta.get('created'),
                'descripcion': oferta.get('description'),
                'tecnologia_consulta': tecnologia,
                'ciudad_consulta': ciudad,
            })
    return pd.DataFrame(filas)

carpetas = sorted(p for p in RAW_ADZUNA.iterdir() if p.is_dir() and p.name[:2] == '20')
adzuna = cargar_adzuna(carpetas[-1])
adzuna.shape

In [ ]:
adzuna.info()
adzuna.head()

## 1.2 Tecnoempleo

In [ ]:
carpetas = sorted(p for p in RAW_TECNOEMPLEO.iterdir() if p.is_dir())
tecnoempleo = pd.read_csv(carpetas[-1] / 'ofertas_tecnoempleo.csv')
tecnoempleo.info()
tecnoempleo.head()

## 1.3 Primeras observaciones

_Anotad aquí lo que veis: columnas inesperadas, tipos incorrectos, nulos llamativos…_